In [ ]:
%run setup.py

# 01 · Análisis Exploratorio de Datos (EDA)

**Escenario 1 · Predicción de precios de viviendas**
Dataset: *Melbourne Housing Market* (Kaggle).

Objetivos del EDA:
1. Verificar que el dataset cumple los requisitos mínimos del proyecto.
2. Conocer la distribución de la variable objetivo (`Price`).
3. Detectar valores nulos, duplicados, valores atípicos y relaciones entre variables.

In [ ]:
# CELDA 1: Carga del dataset
df = load_raw()
print(f"Registros: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
display(df.head())

## 1.1 Tipos de variables y requisitos del dataset

In [ ]:
categoricas = df.select_dtypes(exclude="number").columns.tolist()
numericas = df.select_dtypes(include="number").columns.tolist()

print(f"Variables categóricas ({len(categoricas)}):", categoricas)
print(f"Variables numéricas ({len(numericas)}):", numericas)
print()
print("Requisitos del documento:")
print(f"  - Mínimo 5.000 registros y 13 columnas : {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"  - Al menos 3 categóricas               : {len(categoricas)}")
print(f"  - Al menos 10 numéricas                : {len(numericas)}")

assert df.shape[0] >= 5000 and df.shape[1] >= 13
assert len(categoricas) >= 3 and len(numericas) >= 10
print("\nEl dataset CUMPLE los requisitos mínimos.")

## 1.2 Calidad de los datos: nulos y duplicados

In [ ]:
resumen_nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2),
}).sort_values("nulos", ascending=False)
display(resumen_nulos)
print(f"Total de celdas nulas: {int(df.isna().sum().sum()):,}")
print(f"Filas duplicadas: {int(df.duplicated().sum()):,}")

## 1.3 Estadísticas descriptivas

In [ ]:
display(df.describe().T.round(2))

## 1.4 Distribución de la variable objetivo (`Price`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df["Price"].dropna(), bins=60, color="steelblue", edgecolor="black")
axes[0].set_title("Distribución del precio")
axes[0].set_xlabel("Precio (AUD)")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(np.log1p(df["Price"].dropna()), bins=60, color="seagreen", edgecolor="black")
axes[1].set_title("Distribución del logaritmo del precio")
axes[1].set_xlabel("log(1 + Precio)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

print("El precio presenta asimetría positiva: pocas viviendas muy caras.")
print("Por eso se revisa también su transformación logarítmica.")

## 1.5 Variables categóricas y ubicación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df["Type"].value_counts().plot(kind="bar", ax=axes[0], color="darkorange", edgecolor="black")
axes[0].set_title("Tipo de vivienda (Type)")
axes[0].set_xlabel("Tipo")
axes[0].set_ylabel("Cantidad")

df["Regionname"].value_counts().plot(kind="barh", ax=axes[1], color="teal", edgecolor="black")
axes[1].set_title("Regiones de Melbourne")
axes[1].set_xlabel("Cantidad")

plt.tight_layout()
plt.show()

display(df["Type"].value_counts().to_frame("cantidad"))

## 1.6 Matriz de correlación (variables numéricas)

In [ ]:
plt.figure(figsize=(12, 9))
correlacion = df[numericas].corr(numeric_only=True)
sns.heatmap(correlacion, cmap="coolwarm", center=0, annot=False, linewidths=0.4)
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

print("Correlación con Price (ordenada):")
print(correlacion["Price"].sort_values(ascending=False).round(3))

## 1.7 Valores atípicos y hallazgos

In [ ]:
q1, q3 = df["Price"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
print(f"Rango intercuartílico del precio: Q1={q1:,.0f}  Q3={q3:,.0f}  IQR={iqr:,.0f}")
print(f"Valores por encima del límite superior: {(df['Price'] > limite_superior).sum():,}")
print()
print("Hallazgos principales:")
print("  1. Existen 7.610 precios nulos; se eliminarán en el preprocesamiento.")
print("  2. BuildingArea y YearBuilt tienen >50% de nulos: se imputan con la mediana.")
print("  3. El precio es asimétrico y tiene outliers; se documentan como limitación.")
print("  4. Rooms, BuildingArea y la ubicación son las variables más prometedoras.")